# Bronze → Silver

Limpeza, tipagem e padronização dos dados da Bronze.

- `tb_precos`: um registro por posto, produto e data de coleta, atualizado de forma incremental com MERGE
- `tb_ipca`: variação mensal do IPCA e índice acumulado

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalogo = "preco_na_bomba"
bronze = f"{catalogo}.bronze"
silver = f"{catalogo}.silver"
pasta_checkpoints = f"/Volumes/{catalogo}/bronze/inputs/_checkpoints"

# Datas fora do padrão viram NULL em vez de gerar erro de versão do parser
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver}")

DataFrame[]

## 1. tb_precos

**Regras de limpeza**
- `cnpj`: apenas dígitos; CNPJs sem 14 dígitos são descartados
- `produto`: lista fechada com os 6 combustíveis da pesquisa
- `unidade_medida`: derivada do produto, pois o GNV aparece como `R$ / m³` e `R$ / m3`
- `data_coleta`: convertida do formato `dd/MM/yyyy`
- `valor_venda`: vírgula decimal convertida para ponto; valores fora da faixa de R$ 1 a R$ 20 são descartados
- Textos padronizados: nomes e municípios com iniciais maiúsculas, siglas e bandeiras em maiúsculas
- Linhas sem CNPJ, produto, data ou valor são descartadas (inclui as linhas vazias no fim dos arquivos)

**Carga incremental**
A Silver lê a Bronze como stream e aplica um MERGE a cada lote. Só as linhas novas da Bronze são processadas, e reprocessar um arquivo atualiza os registros em vez de duplicar.

In [0]:
produtos = {
    "GASOLINA": "Gasolina",
    "GASOLINA ADITIVADA": "Gasolina Aditivada",
    "ETANOL": "Etanol",
    "DIESEL": "Diesel",
    "DIESEL S10": "Diesel S10",
    "GNV": "GNV",
}
mapa_produtos = F.create_map([F.lit(x) for par in produtos.items() for x in par])


def limpar_precos(df):
    cnpj = F.regexp_replace(F.col("cnpj"), r"\D", "")
    produto = mapa_produtos[F.upper(F.trim(F.col("produto")))]
    valor_texto = F.regexp_replace(F.trim(F.col("valor_venda")), ",", ".")
    valor = F.when(valor_texto.rlike(r"^\d{1,2}(\.\d{1,3})?$"), valor_texto.cast("decimal(10,3)"))

    return df.select(
        F.when(F.length(cnpj) == 14, cnpj).alias("cnpj"),
        produto.alias("produto"),
        F.to_date(F.try_to_timestamp(F.trim(F.col("data_coleta")), F.lit("dd/MM/yyyy"))).alias("data_coleta"),
        F.when(valor.between(1, 20), valor).alias("valor_venda"),
        F.when(produto == "GNV", "R$ / m³").otherwise("R$ / litro").alias("unidade_medida"),
        F.initcap(F.trim(F.col("revenda"))).alias("revenda"),
        F.upper(F.trim(F.col("bandeira"))).alias("bandeira"),
        F.upper(F.trim(F.col("regiao"))).alias("regiao"),
        F.upper(F.trim(F.col("uf"))).alias("uf"),
        F.initcap(F.trim(F.col("municipio"))).alias("municipio"),
        F.initcap(F.trim(F.col("bairro"))).alias("bairro"),
        F.regexp_replace(F.col("cep"), r"\D", "").alias("cep"),
        "arquivo_origem",
        "ingestion_datetime",
    )

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {silver}.tb_precos (
        cnpj STRING,
        produto STRING,
        data_coleta DATE,
        valor_venda DECIMAL(10,3),
        unidade_medida STRING,
        revenda STRING,
        bandeira STRING,
        regiao STRING,
        uf STRING,
        municipio STRING,
        bairro STRING,
        cep STRING,
        arquivo_origem STRING,
        ingestion_datetime TIMESTAMP
    )
""")


def upsert_precos(batch_df, batch_id):
    # O MERGE falha se duas linhas do lote apontarem para o mesmo registro, então o lote é deduplicado antes
    janela = Window.partitionBy("cnpj", "produto", "data_coleta").orderBy(F.col("ingestion_datetime").desc())

    novos = (
        limpar_precos(batch_df)
        .filter(
            F.col("cnpj").isNotNull()
            & F.col("produto").isNotNull()
            & F.col("data_coleta").isNotNull()
            & F.col("valor_venda").isNotNull()
        )
        .withColumn("rn", F.row_number().over(janela))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

    novos.createOrReplaceTempView("novos_precos")
    novos.sparkSession.sql(f"""
        MERGE INTO {silver}.tb_precos AS t
        USING novos_precos AS n
        ON t.cnpj = n.cnpj AND t.produto = n.produto AND t.data_coleta = n.data_coleta
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)


(
    spark.readStream.table(f"{bronze}.tb_precos_revenda")
    .writeStream
    .foreachBatch(upsert_precos)
    .option("checkpointLocation", f"{pasta_checkpoints}/silver_tb_precos")
    .trigger(availableNow=True)
    .start()
    .awaitTermination()
)

In [0]:
display(spark.sql(f"""
    SELECT produto, unidade_medida,
           COUNT(*) AS linhas,
           COUNT(DISTINCT cnpj) AS postos,
           MIN(valor_venda) AS menor_valor,
           ROUND(AVG(valor_venda), 2) AS valor_medio,
           MAX(valor_venda) AS maior_valor
    FROM {silver}.tb_precos
    GROUP BY produto, unidade_medida
    ORDER BY linhas DESC
"""))

produto,unidade_medida,linhas,postos,menor_valor,valor_medio,maior_valor
Gasolina,R$ / litro,445437,12284,4.550,6.08,9.290
Etanol,R$ / litro,375059,10858,2.630,4.25,7.440
Gasolina Aditivada,R$ / litro,346103,9960,4.690,6.28,9.790
Diesel S10,R$ / litro,310809,9789,4.730,6.11,9.290
Diesel,R$ / litro,188550,5869,4.690,6.06,8.390
GNV,R$ / m³,37181,1024,3.390,4.76,6.890


In [0]:
# Quantas linhas da Bronze cada regra descartou, por arquivo
df_conferencia = limpar_precos(spark.table(f"{bronze}.tb_precos_revenda"))

display(
    df_conferencia.groupBy("arquivo_origem")
    .agg(
        F.count("*").alias("linhas_bronze"),
        F.sum(F.col("cnpj").isNull().cast("int")).alias("sem_cnpj"),
        F.sum(F.col("produto").isNull().cast("int")).alias("sem_produto"),
        F.sum(F.col("data_coleta").isNull().cast("int")).alias("sem_data"),
        F.sum(F.col("valor_venda").isNull().cast("int")).alias("sem_valor"),
    )
    .orderBy("arquivo_origem")
)

arquivo_origem,linhas_bronze,sem_cnpj,sem_produto,sem_data,sem_valor
ca-2024-01.csv,477154,0,0,0,0
ca-2024-02.csv,421382,0,0,0,0
ca-2025-01.csv,429523,9114,9114,9114,9114
ca-2025-02.csv,384208,0,0,0,0


## 2. tb_ipca

- Mantém a versão mais recente de cada mês, já que a Bronze recebe um lote novo a cada execução
- `indice_acumulado`: produto acumulado de (1 + variação/100) desde jan/2024; a Gold usa esse índice para trazer os preços a valores do mês mais recente

In [0]:
janela_ipca = Window.partitionBy("data").orderBy(F.col("ingestion_datetime").desc())

# Série pequena (um registro por mês), então a janela sem partição não pesa
janela_acumulado = Window.orderBy("mes_referencia").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_ipca = (
    spark.table(f"{bronze}.tb_ipca")
    .withColumn("rn", F.row_number().over(janela_ipca))
    .filter(F.col("rn") == 1)
    .select(
        F.to_date(F.try_to_timestamp(F.col("data"), F.lit("dd/MM/yyyy"))).alias("mes_referencia"),
        F.col("valor").cast("decimal(6,2)").alias("variacao_mensal"),
    )
    .withColumn(
        "indice_acumulado",
        F.exp(F.sum(F.log1p(F.col("variacao_mensal") / 100)).over(janela_acumulado)).cast("decimal(12,6)"),
    )
)

df_ipca.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver}.tb_ipca")

display(df_ipca.orderBy("mes_referencia"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


mes_referencia,variacao_mensal,indice_acumulado
2024-01-01,0.42,1.004200
2024-02-01,0.83,1.012535
2024-03-01,0.16,1.014155
2024-04-01,0.38,1.018009
2024-05-01,0.46,1.022692
2024-06-01,0.21,1.024839
2024-07-01,0.38,1.028734
2024-08-01,-0.02,1.028528
2024-09-01,0.44,1.033053
2024-10-01,0.56,1.038838
